# First experiments with LLM calls

The focus of this notebook is to get more familiar with:
* downloading pre-trained language models from open-source repositories (e.g. the Hugging Face Hub)
* instantiating them
* playing with their generation parameters and probing their outputs

The notebook runs on a Google Colab kernel (free tier, T4 GPU with ~15GB of VRAM). To reproduce it, you need a Google account and a connection to a Colab runtime from VS Code.

## Table of contents

1. [Importing modules](#importing-modules)
2. [Setting parameters](#setting-parameters)
3. [Loading the Phi-3 model and tokenizer](#loading-the-phi-3-model-and-tokenizer)
4. [Basic invocation with a system and a user prompt](#basic-invocation-with-a-system-and-a-user-prompt)
5. [Playing with temperature and top_p](#playing-with-temperature-and-top_p)
    * [How the next token is chosen](#how-the-next-token-is-chosen)
    * [Measuring output entropy](#measuring-output-entropy)
    * [Varying temperature](#varying-temperature)
    * [Varying top_p](#varying-top_p)
6. [Takeaways](#takeaways)

## Importing modules

In [ ]:
from transformers import (pipeline, AutoTokenizer, AutoModelForCausalLM)

## Setting parameters

In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

## Loading the Phi-3 model and tokenizer

We load [`microsoft/Phi-3-mini-4k-instruct`](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) (~3.8B parameters), which fits comfortably on the Colab T4 GPU in half precision. Then we wrap the model and tokenizer in a `text-generation` pipeline.

In [3]:
model= AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda",
    torch_dtype="auto",
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=True,
    temperature=0.7,
)

## Basic invocation with a system and a user prompt

The pipeline accepts a list of chat messages. The **system** prompt sets the model's persona and behaviour, and the **user** prompt is the actual request. Because `do_sample=True`, every call can return a different completion.

In [ ]:
SYSTEM_PROMPT = """You are a really dark comedian. 
You don't have any problem with pushing the boundaries of comedy
Your humor is dark, edgy, your punchlines completely unexpected and you are not afraid to make fun of sensitive topics.
Your jokes are short, punchy and sharp."""

USER_PROMPT = "Write a joke about a man who is afraid of flying"
    
messages = [{'role':'system', 'content': SYSTEM_PROMPT}, 
{'role': 'user', 'content': USER_PROMPT}]

In [9]:
for tries in range(5):
    output = generator(messages)
    print(f"Attempt {tries+1}:")
    print(output[0]['generated_text'])
    print("\n" + "-"*50 + "\n")

Attempt 1:
Why did the man sit next to a pilot on the plane? 

Because he thought they might start a revolution with their takeoff technique!

--------------------------------------------------

Attempt 2:
Why did the man bring a ladder on his flight? Because he heard the plane was taking off and he didn't want to miss his chance to be grounded!

--------------------------------------------------

Attempt 3:
Why did the man cross the runway? To get to his fear of flying.

--------------------------------------------------

Attempt 4:
Why did the man who was afraid of flying become a gardener? 

Because he realized that he could literally plant his fears and watch them grow!

--------------------------------------------------

Attempt 5:
Why was the man afraid of flying? Because every time he went to the airport, he was sure he'd meet a pilot who'd throw him off the plane!

--------------------------------------------------



## Playing with temperature and top_p

In this section we look at how the `temperature` and `top_p` parameters affect the output. First the theory, then the experiments.

### How the next token is chosen

At every generation step the model outputs one score (a **logit** $z_i$) for each token $i$ in its vocabulary. When `do_sample=True`, Hugging Face's `generate` turns these scores into a token in a fixed order:

```
logits ──► ÷ temperature ──► softmax ──► top_p filter ──► renormalize ──► sample one token
```

The sampled token is appended to the input, and the whole process repeats until `max_new_tokens` is reached or an end-of-sequence token is produced.

#### 1. `temperature`: reshaping the distribution

The logits are divided by the temperature $T$ before the softmax:

$$
p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}
$$

- **$T < 1$** makes the gaps between logits larger, so most of the probability goes to the top tokens. As $T \to 0$ this becomes greedy decoding (always pick the most likely token).
- **$T = 1$** leaves the model's original distribution unchanged.
- **$T > 1$** makes the gaps smaller, so the distribution gets flatter and unlikely tokens are picked more often.

![Effect of temperature on the next-token distribution](images/temperature.png)

#### 2. `top_p` (nucleus sampling): cutting the tail

After the temperature step, `top_p` limits which tokens can be sampled:

1. Sort the tokens by probability, from highest to lowest.
2. Keep the **smallest** set of top tokens whose probabilities add up to at least `top_p`. This set is the *nucleus*.
3. Set the probability of every other token to zero, rescale the kept ones so they sum to 1, and sample from them.

![Nucleus (top_p) sampling](images/top_p.png)

The size of the nucleus depends on the model's confidence. When one token dominates, the nucleus may contain only that token. When the model is unsure, many tokens get in. (By contrast, `top_k` always keeps exactly $k$ tokens.)

#### How they interact

| Setting | Effect |
|---|---|
| low `temperature` | sharp distribution, so the output is close to deterministic and `top_p` hardly matters |
| high `temperature`, `top_p = 1.0` | flat distribution with the full tail available, so the output is diverse but can go off-topic or contain errors |
| moderate `temperature` (~0.7–1.0), `top_p ≈ 0.9` | some spread, with the junk tail removed, so the output is varied and still coherent |
| very high `temperature` (≫ 1), any `top_p` except very low | the distribution is so flat that even the nucleus is full of junk tokens, so `top_p` can't fully compensate |

Both parameters only have an effect when `do_sample=True`. Common defaults are `temperature ≈ 0.7` and `top_p ≈ 0.9–0.95`.

For the experiments below we use a factual question with a single correct answer, so it's easy to see when the model starts to drift.

### Measuring output entropy

There are several ways to measure how temperature and `top_p` affect the output. Here we use one of the simplest: the **average per-token entropy** of the distribution the model samples from.

At each generation step we take the probabilities *after* temperature and `top_p` have been applied, compute $H_t = -\sum_i p_i \log_2 p_i$ (in bits), and average over all generated tokens. 0 bits means every token choice was certain; 1 bit is like a coin flip between two tokens.

The `pipeline` only returns text, so the helper below calls `model.generate` directly and asks for the per-step scores.

**Expectations:**
- as the temperature rises, the distribution flattens and the average entropy increases;
- at a fixed temperature, lowering `top_p` removes tokens from the distribution, so the average entropy decreases.

In [25]:
import math
import torch

def generate_with_entropy(messages, temperature, top_p=1.0, max_new_tokens=500):
    # Same output as the pipeline, but also return the average entropy of the generated tokens
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        return_dict_in_generate=True,  # return an object, not just token ids
        output_scores=True,            # per-step scores AFTER temperature/top_p
    )

    # Decode only the newly generated tokens (like return_full_text=False)
    prompt_len = inputs["input_ids"].shape[1]
    text = tokenizer.decode(out.sequences[0, prompt_len:], skip_special_tokens=True)

    # out.scores: tuple with one (batch, vocab) tensor per generated token
    scores = torch.stack(out.scores)[:, 0, :].float()   # (steps, vocab)
    probs = torch.softmax(scores, dim=-1)                # tokens filtered by top_p get p = 0
    entropy = torch.special.entr(probs).sum(-1) / math.log(2)  # bits, one per step

    return text, entropy.mean().item()

In [26]:
SYSTEM_PROMPT = """You are a really considerate teacher. 
You provide concise and clear explanations to students.
You are patient and understanding, and you are able to explain complex concepts in a way that is easy to understand.
You are able to provide examples and analogies to help students understand the material."""

USER_PROMPT = "Excuse me sir, can you tell me the name of the first president of the United States?"
    
messages = [{'role':'system', 'content': SYSTEM_PROMPT}, 
{'role': 'user', 'content': USER_PROMPT}]

### Varying temperature

We keep `top_p = 1.0` (no filtering) and sweep the temperature, generating 5 completions per value. For each completion we record its average per-token entropy.

In [29]:
temperature_entropy = {}

for temperature in [.1, .3, .5, .7, 1.0, 2.0, 3.0]:
    print(f"Temperature: {temperature}")
    entropies = []

    for tries in range(5):
        text, h = generate_with_entropy(messages, temperature=temperature)
        entropies.append(h)
        print(f"Attempt {tries+1} (entropy: {h:.3f} bits):")
        print(text)
        print("\n" + "-"*50 + "\n")

    temperature_entropy[temperature] = sum(entropies) / len(entropies)
    print(f"==> Mean entropy at T={temperature}: {temperature_entropy[temperature]:.3f} bits\n")

Temperature: 0.1
Attempt 1 (entropy: 0.005 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

Attempt 2 (entropy: 0.005 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

Attempt 3 (entropy: 0.005 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

Attempt 4 (entropy: 0.005 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

Attempt 5 (entropy: 0.005 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

==> Mean entropy at T=0

In [31]:
print("Temperature sweep:", {k: round(v, 3) for k, v in temperature_entropy.items()})

Temperature sweep: {0.1: 0.005, 0.3: 0.125, 0.5: 0.22, 0.7: 0.271, 1.0: 0.542, 2.0: 4.526, 3.0: 5.385}


As the temperature rises, both the entropy and the variety of the outputs increase:

| Temperature | Mean entropy (bits) | What the outputs look like |
|---|---|---|
| 0.1 | 0.005 | five identical answers (effectively greedy decoding) |
| 0.3–0.7 | 0.13–0.27 | same answer, small variations in wording ("Of course!", "Certainly!") |
| 1.0 | 0.54 | correct answer, but length and amount of extra detail vary a lot |
| 2.0 | 4.53 | still often mentions Washington, but rambling, odd phrasing, made-up facts |
| 3.0 | 5.39 | incoherent text and wrong answers ("James") |

The increase is **not linear**: entropy roughly doubles from 0.7 to 1.0, then jumps about 8x from 1.0 to 2.0. Above $T = 1$ the probability mass spreads into the long tail of the ~32k-token vocabulary, and those tokens start being sampled.

Note that this entropy is measured along each sample's own text. Once the model has produced a few unlikely tokens, it becomes even less certain about what comes next, so high-temperature runs feed on themselves.

### Varying top_p

`top_p` is meant to limit exactly this kind of drift. Instead of sampling from the whole distribution, it keeps only the smallest set of most likely tokens whose cumulative probability reaches `top_p`, and samples from those.

To make the effect easy to see, we fix the temperature at the extreme value of `3.0`, where unfiltered sampling produces gibberish, and sweep `top_p`.

**Expectation:** lower `top_p` values shrink the average entropy, and higher values let it grow back toward the unfiltered value (~5.4 bits).

In [32]:
temperature = 3.0
print("Probaing top-p sampling with temperature = {}".format(temperature))

top_p_entropy = {}

for top_p in [.1, .3, .5, .7, .9, 1.0]:
    print(f"Top-p: {top_p}")
    entropies = []

    for tries in range(5):
        text, h = generate_with_entropy(messages, temperature=temperature, top_p=top_p)
        entropies.append(h)
        print(f"Attempt {tries+1} (entropy: {h:.3f} bits):")
        print(text)
        print("\n" + "-"*50 + "\n")

    top_p_entropy[top_p] = sum(entropies) / len(entropies)
    print(f"==> Mean entropy at top_p={top_p}: {top_p_entropy[top_p]:.3f} bits\n")


Probaing top-p sampling with temperature = 3.0
Top-p: 0.1
Attempt 1 (entropy: 0.000 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

Attempt 2 (entropy: 0.000 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

Attempt 3 (entropy: 0.000 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

Attempt 4 (entropy: 0.000 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

--------------------------------------------------

Attempt 5 (entropy: 0.000 bits):
The first president of the United States was George Washington. He served as president from 1789 to 1797.

----------------------------------

In [33]:
print("top_p sweep (T=3.0):", {k: round(v, 3) for k, v in top_p_entropy.items()})

top_p sweep (T=3.0): {0.1: 0.0, 0.3: 0.393, 0.5: 1.493, 0.7: 3.895, 0.9: 5.13, 1.0: 5.504}


## Takeaways

**Temperature** (with `top_p = 1.0`):
- Average entropy rises steadily up to $T = 1$ (0.005 → 0.54 bits), then jumps sharply (4.5 bits at $T = 2$, 5.4 at $T = 3$).
- Up to $T = 1$ the answer stays correct and only the wording varies. From $T = 2$ the text becomes rambling, then incoherent, and the answer is often wrong.

**top_p** (with `temperature = 3.0`):

| `top_p` | Mean entropy (bits) | What the outputs look like |
|---|---|---|
| 0.1 | 0.000 | five identical, correct answers: the nucleus is a single token at every step, so sampling is greedy *even at T = 3* |
| 0.3 | 0.39 | correct answers with varied wording, similar to T ≈ 0.5–0.7 without filtering |
| 0.5 | 1.49 | mostly correct, but some answers drift into long, invented detail |
| 0.7 | 3.90 | wrong names (John Mason, John Adams) and garbled dates |
| 0.9–1.0 | 5.1–5.5 | gibberish, wrong answers (Abraham Lincoln, Benjamin Franklin) |

- As expected, lowering `top_p` reduces entropy, all the way to 0 at `top_p = 0.1`.
- But `top_p` **cannot rescue a very high temperature**. At $T = 3$ the distribution is so flat that even a 70–90% nucleus contains many junk tokens. `top_p` only cuts the tail; it doesn't restore the shape that temperature flattened.

**In practice**
- `temperature` decides *how peaked* the distribution is, and `top_p` decides *how much of the tail* can be sampled. They work best together at moderate values, not as a way to cancel each other out.
- For factual Q&A: low temperature (≤ 0.7), where `top_p` hardly matters.
- For creative tasks (like the jokes above): temperature around 0.7–1.0 with `top_p ≈ 0.9` gives variety without falling into the tail.
- Caveat: these are 5 samples per setting on a single prompt, so the numbers show trends rather than precise values.